# Peru (PEN) — Fixed Income Derivatives

BTP sovereign bonds and VAC inflation-linked bonds.

**Key features:**
- BTP: Bonos del Tesoro Público (semi-annual, ACT/365)
- VAC: Valor Adquisitivo Constante (IPC-linked)
- PEN curve: quarterly compounding, ACT/360

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.peruvian import (
    BTPPeru, VACBond, build_pen_curve, synthetic_pen_strip,
)
from pricebook.fixed_income.inflation_unit import dual_curve_breakeven
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"BCRP reference rate: 5.25%")

## 1. PEN Discount Curve

In [ ]:
strip = synthetic_pen_strip(REF, rate=0.0525)
pen_curve = build_pen_curve(REF, strip)

print(f"PEN Swap Strip ({len(strip)} points):")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in strip:
    print(f"{c['months']:>6}M  {c['rate']*100:>7.2f}%  {pen_curve.df(c['maturity']):>10.6f}")

with apply_theme():
    fig, axes = create_figure(1)
    ax = axes[0]
    tenors = [c["years"] for c in strip]
    ax.plot(tenors, [c["rate"]*100 for c in strip], 'o-', linewidth=2)
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Rate (%)")
    ax.set_title("PEN Discount Curve — Nov 2024")
    ax.axhline(5.25, color='gray', linestyle='--', alpha=0.5, label='BCRP 5.25%')
    ax.legend()
    fig.tight_layout()

## 2. BTP — Peruvian Sovereign Bond

In [ ]:
print(f"BTP Peru Pricing (semi-annual, ACT/365):")
print(f"{'Coupon':>8}  {'Tenor':>6}  {'Dirty Price':>12}")
print(f"{'─'*8}  {'─'*6}  {'─'*12}")
for cpn in [0.04, 0.05, 0.065]:
    for t in [3, 5, 10, 20, 30]:
        btp = BTPPeru(REF, REF + relativedelta(years=t), cpn)
        px = btp.dirty_price(pen_curve)
        above = "↑" if px > 100 else "↓"
        print(f"{cpn*100:>7.2f}%  {t:>4}Y  {px:>11.4f} {above}")
    print()

## 3. VAC — Inflation-Linked Bond

In [ ]:
# Real curve at ~2%
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod

real_dates = [REF + relativedelta(years=y) for y in [1, 2, 3, 5, 10, 20]]
real_dfs = [math.exp(-0.02 * y) for y in [1, 2, 3, 5, 10, 20]]
real_curve = DiscountCurve(REF, real_dates, real_dfs, interpolation=InterpolationMethod.LOG_LINEAR)

ipc = 120.5  # current IPC value

print(f"VAC Bond Pricing (IPC = {ipc}):")
print(f"{'Tenor':>6}  {'Real Cpn':>10}  {'Real PV':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}")
for t, cpn in [(3, 0.025), (5, 0.03), (10, 0.03), (20, 0.035)]:
    vac = VACBond(REF, REF + relativedelta(years=t), cpn)
    r = vac.price(REF, real_curve, ipc)
    print(f"{t:>4}Y  {cpn*100:>9.2f}%  {r.real_price:>10.4f}")

# BEI
bei = dual_curve_breakeven(pen_curve, real_curve, [1, 2, 5, 10, 20])
print(f"\nBreakeven Inflation (PEN nominal vs real):")
for row in bei:
    print(f"  {row['years']:>2}Y: {row['bei']*100:.2f}%")

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| BTP Peru | Semi-annual, ACT/365 | Peruvian sovereign |
| VAC | Semi-annual, IPC-linked | CPI-adjusted bond |
| BEI | Nominal - Real | ~3.3% implied inflation |